**Building a character level language model using seq2seq networks**

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
with open('/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/data/1268-0.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()

In [3]:
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('End of the Project Gutenberg')

text = text[start_indx:end_indx]

char_set = set(text)

In [4]:
chars_sorted = sorted(char_set)
char2int = {ch:i for i,ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)

In [5]:
text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype = np.int32
)

In [6]:
text_encoded

array([44, 32, 29, ...,  6,  6,  6], dtype=int32)

In [7]:
print('Text encoded shape" ', text_encoded.shape)

Text encoded shape"  (1112310,)


In [8]:
print(text[:15], "==== Encoding ====> ", [char2int[ch] for ch in text[:15]])

THE MYSTERIOUS  ==== Encoding ====>  [44, 32, 29, 1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43, 1]


In [10]:
print(text_encoded[:15], " ==== Reverse =====> ", ''.join(char_array[text_encoded[:15]]))

[44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]  ==== Reverse =====>  THE MYSTERIOUS 


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset

In [12]:
seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size+1)]

In [14]:
from torch.utils.data import Dataset

In [15]:
class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks
    
    def __len__(self):
        return len(self.text_chunks)
    
    def __getitem__(self, index):
        text_chunk = self.text_chunks[index]
        return text_chunk[:-1].long(), text_chunk[1:].long()

In [56]:
len(text_chunks)

1112270

In [61]:
import random

In [62]:
random.shuffle(text_chunks)

In [63]:
train_chunks = text_chunks[:int(0.8*len(text_chunks))]
val_chunks = text_chunks[int(0.8*len(text_chunks)):]

In [64]:
train_seq_dataset = TextDataset(torch.tensor(train_chunks))
val_seq_dataset = TextDataset(torch.tensor(val_chunks))

In [65]:
for i, (seq, target) in enumerate(train_seq_dataset):
    print(' Input(x): ',
    repr(''.join(char_array[seq])))
    print('Target (y): ',
    repr(''.join(char_array[target])))
    print()
    if i == 2:
        break

 Input(x):  ' In general, Pencroft, Herbert, and Neb\n'
Target (y):  'In general, Pencroft, Herbert, and Neb\nw'

 Input(x):  'or too little, so as to cause their cica'
Target (y):  'r too little, so as to cause their cicat'

 Input(x):  'rocks, seems to me to be just the harbor'
Target (y):  'ocks, seems to me to be just the harbor '



In [66]:
batch_size = 128
train_seq_dl = DataLoader(train_seq_dataset, batch_size, shuffle=True)
val_seq_dl = DataLoader(val_seq_dataset, batch_size)

**Building a character level RNN model:**

In [67]:
import torch.nn as nn
from utils.early_stopping import EarlyStopping

In [74]:
torch.manual_seed(1)

class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)
    
    def forward(self, input):
        embedded_input = self.embedding(input)
        out, (hidden,cell) = self.rnn(embedded_input)
        return self.fc(out).permute(0, 2, 1)    

In [75]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512

In [76]:
torch.manual_seed(1)

model = RNN(vocab_size, embed_dim, rnn_hidden_size)
model

RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, num_layers=2, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [77]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

model = model.to(device)

In [78]:
loss_fn = nn.CrossEntropyLoss(reduction='sum')


optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

early_stopper = EarlyStopping(patience=50, checkpoint_path = 'seq2seq.pt', restore_best_weights=True, verbose=True)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=5)

In [79]:
num_epochs = 10000

train_loss = [0] * num_epochs
val_loss = [0] * num_epochs

for epoch in range(num_epochs):

    model.train()
    for x_batch, y_batch in train_seq_dl:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # compute the forward pass
        optimizer.zero_grad()
        out = model(x_batch)
        # compute the loss
        loss = loss_fn(out, y_batch)
        # back propagation
        loss.backward()
        # optimizer
        optimizer.step()
        train_loss[epoch] += loss.item()
    train_loss[epoch] /= len(train_seq_dl.dataset) 
    
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in val_seq_dl:
            # move to GPU
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            # forward pass
            out = model(x_batch)
            loss = loss_fn(out, y_batch)
            val_loss[epoch] += loss.item()
    val_loss[epoch] /= len(val_seq_dl.dataset)

    scheduler.step(val_loss[epoch])

    print(f"Epoch: {epoch+1} | Train Loss: {train_loss[epoch]:.4f} | Val Loss: {val_loss[epoch]:.4f}")
    # early stopping
    early_stopper(val_loss[epoch], model, optimizer, epoch)
    if early_stopper.should_stop:
        print("Stopping at epoch: ", epoch)
        break

Epoch: 1 | Train Loss: 44.6203 | Val Loss: 36.5641
Metric improved to 36.5641. Checkpoint saved at 0
Epoch: 2 | Train Loss: 32.0504 | Val Loss: 28.4944
Metric improved to 28.4944. Checkpoint saved at 1
Epoch: 3 | Train Loss: 25.6881 | Val Loss: 24.2496
Metric improved to 24.2496. Checkpoint saved at 2
Epoch: 4 | Train Loss: 22.3956 | Val Loss: 21.9215
Metric improved to 21.9215. Checkpoint saved at 3
Epoch: 5 | Train Loss: 20.6493 | Val Loss: 20.6514
Metric improved to 20.6514. Checkpoint saved at 4
Epoch: 6 | Train Loss: 19.6209 | Val Loss: 19.9742
Metric improved to 19.9742. Checkpoint saved at 5
Epoch: 7 | Train Loss: 18.9379 | Val Loss: 19.4601
Metric improved to 19.4601. Checkpoint saved at 6
Epoch: 8 | Train Loss: 18.4619 | Val Loss: 18.9509
Metric improved to 18.9509. Checkpoint saved at 7
Epoch: 9 | Train Loss: 18.1016 | Val Loss: 18.7636
Metric improved to 18.7636. Checkpoint saved at 8
Epoch: 10 | Train Loss: 17.8177 | Val Loss: 18.5088
Metric improved to 18.5088. Checkpoint 

KeyboardInterrupt: 

**stopped it training earlier before it had finished, don't have that much compute and time to blow:**

In [80]:
print('The Island: ', [char2int[ch] for ch in 'The Island'])

The Island:  [44, 57, 54, 1, 33, 68, 61, 50, 63, 53]


In [83]:
island_encoded = torch.tensor([char2int[ch] for ch in 'The Island']).view(1, -1).to(device)

In [95]:
model.eval()

with torch.no_grad():
    text = 'The Island'
    for _ in range(500):
        text_encoded = torch.tensor([char2int[ch] for ch in text]).view(1, -1).to(device)
        out = model(text_encoded)
        char = torch.argmax(out[0, :, -1]).item()
        text += char_array[char]

In [97]:
print(text)

The Island was situated on the thirty-seventh degree
of the southern latitude, or taking into consideration through the
imperfection of the performance, an error of five degrees, that it must
be situated between the thirty-fifth and the fortieth parallel.

There was only the longitude to be obtained, and the position of the
island would be determined, The engineer hoped to attempt this the same
day, at twelve o’clock, at which moment the sun would pass the meridian.

It was decided that Sunday should be s


In [102]:
model.eval()

with torch.no_grad():
    text = 'I saw her'
    for _ in range(500):
        text_encoded = torch.tensor([char2int[ch] for ch in text]).view(1, -1).to(device)
        out = model(text_encoded)
        char = torch.argmax(out[0, :, -1]).item()
        text += char_array[char]

In [103]:
print(text)

I saw here, Ayrton, whose influence has once more manifested itself, who does
for us all that which we cannot do for ourselves, and who, his will
accomplished, conceals himself from us.”

“Let us make search for him, then!” exclaimed Pencroft.

“Yes, we will search for him,” answered Harding, “but we shall not
discover this powerful being who performs such wonders, until he pleases
to call us to him!”

This invisible protection, which rendered their own action unavailing,
both irritated and piqued the en


In [137]:
model.eval()
temperature = 0.01
with torch.no_grad():
    text = 'I saw her'
    for _ in range(500):
        text_encoded = torch.tensor([char2int[ch] for ch in text]).view(1, -1).to(device)
        out = model(text_encoded)
        #char = torch.argmax(out[0, :, -1]).item()
        probs = F.softmax(out[0, :, -1]/temperature, dim=-1)
        char = torch.multinomial(probs, num_samples=1).item()
        text += char_array[char]

In [138]:
print(text)

I saw here, Ayrton, whose influence has once more manifested itself, who does
for us all that which we cannot do for ourselves, and who, his will
accomplished, conceals himself from us.”

“Let us make search for him, then!” exclaimed Pencroft.

“Yes, we will search for him,” answered Harding, “but we shall not
discover this powerful being who performs such wonders, until he pleases
to call us to him!”

This invisible protection, which rendered their own action unavailing,
both irritated and piqued the en


**Adding temperature to add more probabilistic and creative behaviour to the model:**

In [141]:
model.eval()
temperature = 0.4
with torch.no_grad():
    text = 'The storm'
    for _ in range(500):
        text_encoded = torch.tensor([char2int[ch] for ch in text]).view(1, -1).to(device)
        out = model(text_encoded)
        #char = torch.argmax(out[0, :, -1]).item()
        probs = F.softmax(out[0, :, -1]/temperature, dim=-1)
        char = torch.multinomial(probs, num_samples=1).item()
        text += char_array[char]

In [142]:
print(text)

The storm did not seem to have gone
farther to the west. It appeared to have exhausted itself. Could it have
passed away in electric sheets, as is sometimes the case with regard to
the typhoons of the Indian Ocean?

But at the same time, it was also evident that the balloon was again
slowly descending with a regular movement. It appeared as if it were,
little by little, collapsing, and that its case was lengthening and
extending, passing from a spherical to an oval form. Towards midday the
balloon was ho


In [145]:
model.eval()
temperature = 1
with torch.no_grad():
    text = 'The storm'
    for _ in range(500):
        text_encoded = torch.tensor([char2int[ch] for ch in text]).view(1, -1).to(device)
        out = model(text_encoded)
        #char = torch.argmax(out[0, :, -1]).item()
        probs = F.softmax(out[0, :, -1]/temperature, dim=-1)
        char = torch.multinomial(probs, num_samples=1).item()
        text += char_array[char]

In [146]:
print(text)

The storm was then in all its violence, and perhaps at its height. Not a
single ray of light from the moon pierced through the clouds. To follow
a straight course was difficult. It was best to rely on its summit were only, had
a more complete knowledge of hydrogehed, the sail he had gone
somewhat apart, he said,--

“The truth is, my friends, we must not conceal from ourselves that an
important change is going forward. The volcanic substances are no longer
in a state of ebullition, they have caught fire.



The above text generation technique used is called greedy decoding. Often leads to the same words being repeated over and over again. Instead, we can sample the next character randomly, using the model's estimated probability distribution: if the model estimates a probability p for a given token, then this token will be sampled with a probability p. This process generates more diverse and interesting text since the most likely token won't always be sampled. To sample the next token, we use the torch.multinomial function, which samples random class indices, given a list of class probabilities

To generate more convincing text, a common tehcnique is to sample only from the top k characters, or only from the smallest set of top characters whose top probabulity exceeds some threshold. This is called top-sampling or nucleus sampling. 